In [3]:
!pip install transformers datasets torchaudio librosa soundfile jiwer tqdm

In [4]:
# Updated paths from Colab (/content/) to Kaggle (/kaggle/input/) format.
# Dataset is mounted at /kaggle/input/datasets/ravikantgupta123/question-2/asr_project/
# Output directory changed to /kaggle/working/ which is writable on Kaggle.
# 28-03-2026

BASE = "/kaggle/input/datasets/ravikantgupta123/question-2/asr_project"

TRAIN_PATH = f"{BASE}/train.jsonl"
VAL_PATH = f"{BASE}/val.jsonl"
AUDIO_DIR = f"{BASE}/audio"
OUTPUT_DIR = "/kaggle/working"

In [5]:
import json
import pandas as pd

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
      return pd.DataFrame(json.loads(line) for line in f if line.strip())

train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
train_df.head()

,audio_path,text,user_id,duration
0,processed/audio/825727_2.wav,जी जी जी ।,291038,3.33
1,processed/audio/825727_3.wav,जी जी जी,291038,4.02
2,processed/audio/825727_4.wav,हां उधर हां,291038,1.56
3,processed/audio/825727_6.wav,"हा हा,पहली बार था, पहली बार था,क्या आप लोग का ...",291038,13.95
4,processed/audio/825727_7.wav,जी जी जी,291038,2.10


#Fix audio paths

In [6]:
# Updated BASE_AUDIO_DIR to use the AUDIO_DIR variable pointing to Kaggle input dataset location.
# 28-03-2026
import os

BASE_AUDIO_DIR = AUDIO_DIR

def fix_audio_path(path):
    filename = os.path.basename(path)  # → 825780_0.wav
    full_path = os.path.join(BASE_AUDIO_DIR, filename)

    # skip if file missing (prevents crash)
    if not os.path.exists(full_path):
        return None
    return full_path

val_df["audio_path"] = val_df["audio_path"].apply(fix_audio_path)
train_df["audio_path"] = train_df["audio_path"].apply(fix_audio_path)

# remove missing files safely
val_df = val_df[val_df["audio_path"].notnull()].reset_index(drop=True)
train_df = train_df[train_df["audio_path"].notnull()].reset_index(drop=True)

###Load Whisper-small (optimized GPU)
Purpose: faster inference

In [7]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import librosa

# Load model + processor (more stable than pipeline)
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [8]:
def transcribe_audio(path):
  try:
    audio, sr = librosa.load(path, sr=16000)

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
      predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

  except Exception as e:
    return ""

In [9]:
from tqdm import tqdm

def run_asr(df):
    results = []
    for path in tqdm(df["audio_path"]):
      results.append(transcribe_audio(path))
    return results

val_df["raw_hyp"] = run_asr(val_df)

  0%|          | 0/469 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cus

In [11]:
# Changed model and processor save path from Google Drive to /kaggle/working/.
# Kaggle saves everything in /kaggle/working/ as notebook output.
# 28-03-2026
SAVE_PATH = f"{OUTPUT_DIR}/whisper_small_saved"

model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/whisper_small_saved


Saved val_with_asr.jsonl


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="/kaggle/working/whisper_small_saved",
    repo_id="ravikantgupta123/whisper-small-saved",
    repo_type="model",
    token=os.environ.get("")
)
print("Uploaded to HuggingFace!")